In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import decoupler as dc
import os
import gseapy
import matplotlib.pyplot as plt
import squidpy as sq
import pickle
import anndata as ad
import re
from glob import glob
import copy

adata_infile = "data.h5ad"


adata = sc.read_h5ad(adata_infile)

mapping_file = "sample_metadata.csv"

# ----------------------------
# 2️⃣ Load mapping file
# ----------------------------
mapping_df = pd.read_csv(mapping_file, dtype=str)
mapping_df = mapping_df.loc[:, ~mapping_df.columns.duplicated()]

IGNORE_PREFIX_N = 0

cell_type_colors = {
    # --- Malignant epithelial ---
    'CEACAM-high tumor epithelial cells': '#6BA4F8',   # softened azure blue
    'Cycling Tumor Cells': '#F4BA63',                 # softened amber
    'Mucin-producing tumor cells': '#E59973',         # softened coral
    'Inflamed primary tumor epithelial cells': '#FF6259', # softened red

    # --- Tumor microenvironment ---
    'Complement immunosuppressive macrophages (TAMs)': '#874284',  # softened purple
    'Systemic inflammatory macrophage program (TAMs)': '#61385B',   # softened plum
    'CAFs (Cancer associated fibroblasts)': '#9A5766',              # softened burgundy
    'Pericyte-enriched endothelial cells': '#4F709D',              # softened navy

    # --- Immune cells ---
    'Cytotoxic T cells': '#998CFA',   # softened lavender
    'Plasma Cells': '#56B356',        # softened green
}


In [ ]:
"""
Cohort-level paired gene expression analysis (Primary EAC vs Brain Met)
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy import stats

# ═════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════════════

genes_of_interest = ["ERBB2", "EGFR", "ERBB3", "KRAS"]

GROUP_ORDER = ["Primary EAC", "EAC Brain Met"]

MALIGNANT_LABELS = [
    "CEACAM-high tumor epithelial cells",
    "Cycling Tumor Cells",
    "Mucin-producing tumor cells",
    "Inflamed primary tumor epithelial cells",
]

LABEL_FS = 12
TITLE_FS = 12
TICK_FS  = 11
DOT_S    = 30

OUTPUT_DIR = "docs/output"
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")

os.makedirs(PLOT_DIR, exist_ok=True)


# ═════════════════════════════════════════════════════════════════════════════
# SAMPLE METADATA
# ═════════════════════════════════════════════════════════════════════════════

def normalize_acc1(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    return re.sub(r"[^\w\-]", "", s)


def build_sample_metadata(adata, mapping_file, sample_col="sample"):
    mapping_df = pd.read_csv(mapping_file, dtype=str)
    mapping_df["Acc1_key"] = mapping_df["Acc1"].apply(normalize_acc1)

    rows = []
    for sample in adata.obs[sample_col].unique():
        match = mapping_df[mapping_df["Acc1_key"] == normalize_acc1(sample)]

        if match.empty:
            rows.append({
                "sample": sample,
                "base_pid": None,
                "Tumor Location": "unknown",
            })
        else:
            r = match.iloc[0]
            pid = re.search(r"(P-\d+)", str(r.get("Updated_ID-Dec2025", "")))

            rows.append({
                "sample": sample,
                "base_pid": pid.group(1) if pid else None,
                "Tumor Location": r.get("Tumor Location", "unknown"),
            })

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# DATA EXTRACTION
# ═════════════════════════════════════════════════════════════════════════════

def extract_malignant_expression(adata, genes, meta, sample_col="sample"):
    mask = adata.obs["cell_type"].isin(MALIGNANT_LABELS)
    adata_m = adata[mask]

    gene_idx = {g: adata.var_names.get_loc(g) for g in genes}
    meta_map = meta.set_index("sample")

    rows = []

    for sample in adata_m.obs[sample_col].unique():

        if sample not in meta_map.index:
            continue

        m = meta_map.loc[sample]

        if m["Tumor Location"] not in GROUP_ORDER:
            continue

        ad_s = adata_m[adata_m.obs[sample_col] == sample]

        row = {
            "sample": sample,
            "base_pid": m["base_pid"],
            "Tumor Location": m["Tumor Location"],
        }

        X = ad_s.layers.get("lognormal", ad_s.X)

        for g in genes:
            idx = gene_idx[g]
            expr = X[:, idx].toarray().ravel() if hasattr(X, "toarray") else X[:, idx]
            row[f"{g}_mean"] = float(np.mean(expr))

        rows.append(row)

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# FILTER PAIRED PATIENTS
# ═════════════════════════════════════════════════════════════════════════════

def filter_paired(df):
    counts = df.groupby(["base_pid", "Tumor Location"])["sample"].nunique().unstack(fill_value=0)

    valid = counts[(counts.get(GROUP_ORDER[0], 0) > 0) &
                   (counts.get(GROUP_ORDER[1], 0) > 0)].index

    return df[df["base_pid"].isin(valid)].copy()


# ═════════════════════════════════════════════════════════════════════════════
# PLOTTING HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def style(ax):
    ax.set_xticks([0, 1])
    ax.set_xticklabels(GROUP_ORDER, fontsize=TICK_FS)
    ax.set_xlim(-0.3, 1.3)
    ax.spines[["top", "right"]].set_visible(False)


def pvalue(a, b):
    if len(a) < 3:
        return np.nan
    return stats.wilcoxon(a, b).pvalue


def sig(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — PAIRED LINES
# ═════════════════════════════════════════════════════════════════════════════

def plot_fig1(df):
    fig, axes = plt.subplots(1, len(genes_of_interest), figsize=(14, 4), sharey=True)

    if len(genes_of_interest) == 1:
        axes = [axes]

    pvals = {}

    for ax, g in zip(axes, genes_of_interest):
        col = f"{g}_mean"

        prim = df[df["Tumor Location"] == GROUP_ORDER[0]]
        met  = df[df["Tumor Location"] == GROUP_ORDER[1]]

        for pid in df["base_pid"].dropna().unique():
            pvals_pid = prim[prim["base_pid"] == pid][col].values
            mvals_pid = met[met["base_pid"] == pid][col].values

            for p in pvals_pid:
                for m in mvals_pid:
                    ax.plot([0, 1], [p, m], alpha=0.3)

        ax.scatter([0]*len(prim), prim[col], label="Primary")
        ax.scatter([1]*len(met),  met[col],  label="Met")

        pvals[g] = pvalue(
            df[df["Tumor Location"] == GROUP_ORDER[0]][col],
            df[df["Tumor Location"] == GROUP_ORDER[1]][col],
        )

        ax.set_title(f"{g} ({sig(pvals[g])})", fontsize=TITLE_FS)
        style(ax)

    plt.tight_layout()
    fig.savefig(f"{PLOT_DIR}/fig1_paired.png", dpi=200)
    plt.close(fig)

    return pvals


# ═════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — MEAN ± SEM
# ═════════════════════════════════════════════════════════════════════════════

def plot_fig2(df, pvals):
    fig, axes = plt.subplots(1, len(genes_of_interest), figsize=(14, 4))

    if len(genes_of_interest) == 1:
        axes = [axes]

    for ax, g in zip(axes, genes_of_interest):
        col = f"{g}_mean"

        prim = df[df["Tumor Location"] == GROUP_ORDER[0]][col]
        met  = df[df["Tumor Location"] == GROUP_ORDER[1]][col]

        means = [prim.mean(), met.mean()]
        sems  = [prim.sem(), met.sem()]

        ax.errorbar([0, 1], means, yerr=sems, fmt="o-")

        ax.set_title(f"{g} ({sig(pvals[g])})", fontsize=TITLE_FS)
        style(ax)

    plt.tight_layout()
    fig.savefig(f"{PLOT_DIR}/fig2_mean_sem.png", dpi=200)
    plt.close(fig)


# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════

def main(adata, mapping_file):

    meta = build_sample_metadata(adata, mapping_file)

    df = extract_malignant_expression(adata, genes_of_interest, meta)
    df = filter_paired(df)

    df.to_csv(f"{OUTPUT_DIR}/sample_expression.csv", index=False)

    pvals = plot_fig1(df)
    plot_fig2(df, pvals)

    print("Done →", PLOT_DIR)


# ═════════════════════════════════════════════════════════════════════════════
# ENTRY
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("Run main(adata, mapping_file)")

In [ ]:
import os
import re
import numpy as np
import pandas as pd

# =============================================================================
# CONFIG
# =============================================================================

CATEGORY_ORDER = ["copy number gain"]

CAT_COLORS = {
    "ecDNA": "#b5585b",
    "HSR": "#BEA2FE",
    "copy number gain": "#2ca02c",
    "subclonal gain": "#e58d71",
}

# =============================================================================
# HELPERS (previous pipeline)
# =============================================================================
# normalize_acc1()
# match_sample_acc1()
# extract_expression()
# SAMPLE_COL


# =============================================================================
# PRIMARY VS BRAIN MET OVERLAY PLOT
# =============================================================================

def plot_primary_vs_brainmet_overlay(
    primary_df,
    primary_cat_dfs,
    brain_df,
    brain_cat_dfs,
    plot_dir,
    gene="ERBB2",
    use_lognorm=True,
):
    os.makedirs(plot_dir, exist_ok=True)

    col = f"{gene}_{'lognorm' if use_lognorm else 'raw'}"

    bins = np.linspace(0, 9.5, 80) if use_lognorm else np.linspace(0, 500, 80)
    xs = np.linspace(bins[0], bins[-1], 600)

    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
    legend_handles = []

    for cat in CATEGORY_ORDER:

        # ---------------- PRIMARY ----------------
        if cat in primary_cat_dfs:
            vals = primary_cat_dfs[cat][col].values
            vals = vals[vals > 0]

            if len(vals) > 5:
                color = CAT_COLORS.get(cat, "#999999")

                ax.hist(vals, bins=bins, density=True,
                        color=color, alpha=0.08, edgecolor="none")

                kde = gaussian_kde(vals, bw_method=0.08)
                ax.plot(xs, kde(xs), color=color, lw=3, ls="-")

                ax.axvline(vals.mean(), color=color, lw=1, alpha=0.5)

                legend_handles.append(
                    plt.Line2D([0], [0], color=color, lw=3,
                               label=f"{cat} Primary (n={len(vals)})")
                )

        # ---------------- BRAIN MET ----------------
        if cat in brain_cat_dfs:
            vals = brain_cat_dfs[cat][col].values
            vals = vals[vals > 0]

            if len(vals) > 5:
                color = CAT_COLORS.get(cat, "#999999")

                ax.hist(vals, bins=bins, density=True,
                        color=color, alpha=0.05, edgecolor="none")

                kde = gaussian_kde(vals, bw_method=0.08)
                ax.plot(xs, kde(xs), color=color, lw=3, ls="--")

                ax.axvline(vals.mean(), color=color, lw=1, ls="--", alpha=0.5)

                legend_handles.append(
                    plt.Line2D([0], [0], color=color, lw=3, ls="--",
                               label=f"{cat} Brain Met (n={len(vals)})")
                )

    ax.set_xlabel(f"{gene} expression", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"{gene} distribution: Primary vs Brain Met", fontsize=14)

    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(handles=legend_handles, fontsize=9, frameon=False)

    fig.savefig(os.path.join(plot_dir, f"{gene}_overlay.png"), dpi=200, bbox_inches="tight")
    plt.close(fig)

    print("Overlay plot saved.")


# =============================================================================
# METADATA BUILDER (CLEANED)
# =============================================================================

def build_sample_metadata_multi(
    adata,
    mapping_file,
    amp_category_file,
    location_labels,
    sample_col="sample",
):
    mapping_df = pd.read_csv(mapping_file, dtype=str)
    mapping_df["Acc1_key"] = mapping_df["Acc1"].apply(normalize_acc1)

    amp_df = pd.read_csv(amp_category_file, dtype=str)
    amp_df["id_key"] = amp_df["ID"].str.strip().str.upper()
    amp_lookup = amp_df.set_index("id_key")["Amp_category"].to_dict()

    rows = []

    for sample in adata.obs[sample_col].unique():

        row = match_sample_acc1(sample, mapping_df)
        if row is None:
            continue

        if str(row["Tumor Location"]).strip() not in location_labels:
            continue

        mapped_id = row["Updated_ID-Dec2025"]
        pid = re.search(r"(P-\d+)", str(mapped_id))
        base_pid = pid.group(1) if pid else None

        amp = amp_lookup.get((base_pid or "").upper(), "No amp")

        if amp not in CATEGORY_ORDER:
            continue

        rows.append({
            "sample": sample,
            "base_pid": base_pid,
            "Tumor Location": row["Tumor Location"],
            "category": amp,
        })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No samples found for: {location_labels}")

    return df


# =============================================================================
# MAIN PIPELINE 
# =============================================================================

def main_combined(
    adata,
    mapping_file,
    amp_category_file,
    primary_labels={"Primary EAC"},
    brain_met_labels={"EAC Brain Met"},
    output_dir="output_primary_vs_brainmet",
):

    os.makedirs(output_dir, exist_ok=True)

    # ---------------- PRIMARY ----------------
    print("Building primary metadata...")
    primary_meta = build_sample_metadata_multi(
        adata, mapping_file, amp_category_file, primary_labels
    )

    print(f"Primary samples: {primary_meta['sample'].nunique()}")

    primary_df = extract_expression(adata, primary_meta)
    primary_df["cohort"] = "Primary"

    primary_cat_dfs = {
        c: primary_df[primary_df["category"] == c]
        for c in primary_df["category"].unique()
    }

    # ---------------- BRAIN MET ----------------
    print("Building brain met metadata...")
    brain_meta = build_sample_metadata_multi(
        adata, mapping_file, amp_category_file, brain_met_labels
    )

    print(f"Brain met samples: {brain_meta['sample'].nunique()}")

    brain_df = extract_expression(adata, brain_meta)
    brain_df["cohort"] = "BrainMet"

    brain_cat_dfs = {
        c: brain_df[brain_df["category"] == c]
        for c in brain_df["category"].unique()
    }

    # ---------------- SAVE ----------------
    combined = pd.concat([primary_df, brain_df], ignore_index=True)

    csv_path = os.path.join(output_dir, "combined_expression.csv")
    combined.to_csv(csv_path, index=False)

    print(f"Saved → {csv_path}")
    print(f"Primary cells: {len(primary_df)} | Brain met cells: {len(brain_df)}")

    # ---------------- PLOT ----------------
    plot_primary_vs_brainmet_overlay(
        primary_df,
        primary_cat_dfs,
        brain_df,
        brain_cat_dfs,
        plot_dir=output_dir,
        gene="ERBB2",
    )

    print("Done.")


# =============================================================================
# ENTRY
# =============================================================================
# main_combined(adata, mapping_file, "ID-by-ERBB2-amp-category.csv")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import gaussian_kde

# ═════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

GENE = "ERBB2"

GROUP_ORDER = ["Primary EAC", "EAC Brain Met"]

AMP_CATEGORIES = ["HSR", "ecDNA"]

CAT_COLORS = {
    "ecDNA": "#b5585b",
    "HSR": "#BEA2FE",
}

MALIGNANT_LABELS = [
    "CEACAM-high tumor epithelial cells",
    "Cycling Tumor Cells",
    "Mucin-producing tumor cells",
    "Inflamed primary tumor epithelial cells",
]

SAMPLE_COL = "sample"

# Style
LABEL_FS = 16
TITLE_FS = 16
TICK_FS = 16
LINE_LW = 1.4
LINE_ALPHA = 0.6


# ═════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ═════════════════════════════════════════════════════════════════════════════

def normalize_acc1(s, ignore_n=0):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\-]", "", s)
    return s[ignore_n:]


def match_sample_acc1(sample_name, mapping_df):
    key = normalize_acc1(sample_name)

    exact = mapping_df[mapping_df["Acc1_key"] == key]
    if not exact.empty:
        return exact.iloc[0]

    fuzzy = mapping_df[
        mapping_df["Acc1_key"].apply(lambda x: x in key or key in x)
    ]
    if not fuzzy.empty:
        return fuzzy.iloc[0]

    print(f"[WARN] No mapping for sample: {sample_name}")
    return None


# ═════════════════════════════════════════════════════════════════════════════
# METADATA BUILDING
# ═════════════════════════════════════════════════════════════════════════════

def build_sample_metadata(adata, mapping_file, amp_file, sample_col=SAMPLE_COL):
    mapping = pd.read_csv(mapping_file, dtype=str).drop_duplicates()
    mapping["Acc1_key"] = mapping["Acc1"].apply(normalize_acc1)
    mapping["mapped_ID_raw"] = mapping["Updated_ID-Dec2025"]

    amp = pd.read_csv(amp_file, dtype=str)
    amp.columns = amp.columns.str.strip()
    amp["id_key"] = amp["ID"].str.strip().str.upper()
    amp_lookup = amp.set_index("id_key")["Amp_category"].to_dict()

    rows = []

    for sample in adata.obs[sample_col].unique():
        m = match_sample_acc1(sample, mapping)

        if m is None:
            continue

        mapped_id = m["mapped_ID_raw"]
        pid = re.search(r"(P-\d+)", mapped_id)
        base_pid = pid.group(1) if pid else None

        key = base_pid.strip().upper() if base_pid else ""
        amp_cat = amp_lookup.get(key, "No amp")

        rows.append({
            "sample": sample,
            "mapped_ID_raw": mapped_id,
            "base_pid": base_pid,
            "Tumor Location": m["Tumor Location"],
            "category": amp_cat,
        })

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# EXPRESSION EXTRACTION
# ═════════════════════════════════════════════════════════════════════════════

def extract_expression(adata, sample_meta):
    if GENE not in adata.var_names:
        raise ValueError(f"{GENE} not in adata")

    gene_idx = adata.var_names.get_loc(GENE)

    mask = adata.obs["cell_type"].isin(MALIGNANT_LABELS)
    adata_mal = adata[mask]

    sample_map = sample_meta.set_index("sample")

    rows = []

    for sample in adata_mal.obs[SAMPLE_COL].unique():
        if sample not in sample_map.index:
            continue

        meta = sample_map.loc[sample]

        if meta["Tumor Location"] not in GROUP_ORDER:
            continue

        if meta["category"] not in AMP_CATEGORIES:
            continue

        adata_s = adata_mal[adata_mal.obs[SAMPLE_COL] == sample]
        expr = adata_s.layers["lognormal"][:, gene_idx].toarray().ravel()

        rows.append({
            "sample": sample,
            "base_pid": meta["base_pid"],
            "Tumor Location": meta["Tumor Location"],
            "category": meta["category"],
            f"{GENE}_mean": float(np.mean(expr)),
        })

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# FILTERING
# ═════════════════════════════════════════════════════════════════════════════

def filter_paired(df):
    out = []

    for cat in AMP_CATEGORIES:
        sub = df[df["category"] == cat]

        counts = (
            sub.groupby(["base_pid", "Tumor Location"])["sample"]
            .nunique()
            .unstack(fill_value=0)
        )

        paired = counts[
            (counts.get(GROUP_ORDER[0], 0) > 0) &
            (counts.get(GROUP_ORDER[1], 0) > 0)
        ].index

        out.append(sub[sub["base_pid"].isin(paired)])

    return pd.concat(out, ignore_index=True)


# ═════════════════════════════════════════════════════════════════════════════
# STATS
# ═════════════════════════════════════════════════════════════════════════════

def wilcoxon_p(df, cat):
    sub = df[df["category"] == cat]

    pt = (
        sub.groupby(["base_pid", "Tumor Location"])[f"{GENE}_mean"]
        .mean()
        .unstack()
        .dropna()
    )

    a = pt[GROUP_ORDER[0]].values
    b = pt[GROUP_ORDER[1]].values

    if len(a) < 4 or np.all(a == b):
        _, p = stats.ttest_rel(a, b)
    else:
        _, p = stats.wilcoxon(a, b)

    return p


# ═════════════════════════════════════════════════════════════════════════════
# PLOTTING
# ═════════════════════════════════════════════════════════════════════════════

def plot(df, outdir):
    os.makedirs(outdir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(7, 5))

    x = [0, 1]

    for cat in AMP_CATEGORIES:
        sub = df[df["category"] == cat]
        c = CAT_COLORS[cat]

        p = sub[sub["Tumor Location"] == GROUP_ORDER[0]][f"{GENE}_mean"]
        m = sub[sub["Tumor Location"] == GROUP_ORDER[1]][f"{GENE}_mean"]

        means = [p.mean(), m.mean()]
        sems = [stats.sem(p), stats.sem(m)]

        ax.plot(x, means, marker="o", color=c, label=cat)
        ax.errorbar(x, means, yerr=sems, fmt="none", color=c, alpha=0.4)

    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_ORDER)
    ax.set_ylabel(f"{GENE} mean expression")

    ax.legend(frameon=False)

    out = os.path.join(outdir, f"{GENE}_amp_overlay.png")
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"[SAVED] {out}")


# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════

def main(adata, mapping_file, amp_file, output_dir):
    meta = build_sample_metadata(adata, mapping_file, amp_file)
    expr = extract_expression(adata, meta)
    expr = filter_paired(expr)

    expr.to_csv(os.path.join(output_dir, "expression_paired.csv"), index=False)

    plot(expr, output_dir)

    print("[DONE]")


if __name__ == "__main__":
    print("Load adata and call main(...)")
